In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

##testing

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_split_balance_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema =   ("SplitBalanceId string,"
        #             "SourceSystem string,"
        #             "PortfolioKey string,"
        #             "PortfolioName string,"
        #             "IngestionTimestamp timestamp,"
        #             "BatchId string,"
        #             "SplitBalanceKey string,"
        #             "ExposureKey string,"
        #             "TotalRemainingAmount string,"
        #             "FirstPeriodAmount string,"
        #             "FirstPeriodRemainingAmount string,"
        #             "FirstPeriodStartDate string,"
        #             "FirstPeriodEndDate string,"
        #             "FirstPeriodNumberOfInstallements string,"
        #             "SecondPeriodAmount string,"
        #             "SecondPeriodRemainingAmount string,"
        #             "SecondPeriodStartDate string,"
        #             "SecondPeriodEndDate string,"
        #             "SecondPeriodNumberOfInstallements string,"
        #             "BusinessDate date,"
        #             )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.SplitBalance"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ============================== loan_details ====================
        loan_details_extra_filters = None
        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter
        loan_details_selected_fields = [
            "prft_acc_unit",
            "ACCOUNT_NUMBER",
            "prft_acc_type",
            "prft_acc_sn",
            "ACC_STATUS",
            "BusinessDate",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        loan_details_final = loan_details.filter(loan_details.ACC_STATUS != "3")

        # ============================== user_defined_fields ====================
        user_defined_fields_extra_filters = None
        if user_defined_fields_extra_filters is not None:
            user_defined_fields_extra_filters.append(date_filter)
            user_defined_fields_final_filters = " AND ".join(
                user_defined_fields_extra_filters
            )
        else:
            user_defined_fields_final_filters = date_filter
        user_defined_fields_selected_fields = [
            "UD_ACCOUNT_NUMBER",
            "PFG_TAG_SET_CODE",
            "PFG_TAG",
            "FIELD_VALUE",
        ]

        user_defined_fields = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_user_defined_fields",
            user_defined_fields_final_filters,
            user_defined_fields_selected_fields,
        )

        user_defined_fields_UD1 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG500")
                & (user_defined_fields.FIELD_VALUE != "")
            )
            .select(user_defined_fields.UD_ACCOUNT_NUMBER)
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
        )

        user_defined_fields_UD2 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG501")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "SYN_YPOL_KEF")
        )

        user_defined_fields_UD3 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG502")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "KEF_1ST_PERIOD")
        )

        user_defined_fields_UD4 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG503")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "YPOL_KEF_1ST_PERIOD")
        )

        user_defined_fields_UD5 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG504")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "START_DATE_1ST_PERIOD")
        )

        user_defined_fields_UD6 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG505")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "END_DATE_1ST_PERIOD")
        )

        user_defined_fields_UD7 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG506")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "AR_DOSEON_1ST_PERIOD")
        )

        user_defined_fields_UD8 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG507")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "KEF_2ND_PERIOD")
        )

        user_defined_fields_UD9 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG508")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "YPOL_KEF_2ND_PERIOD")
        )

        user_defined_fields_UD10 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG509")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "START_DATE_2ND_PERIOD")
        )

        user_defined_fields_UD11 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG510")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "END_DATE_2ND_PERIOD")
        )

        user_defined_fields_UD12 = (
            user_defined_fields.filter(
                (user_defined_fields.PFG_TAG_SET_CODE == "SPLIT BALANCE")
                & (user_defined_fields.PFG_TAG == "PFTAG511")
            )
            .select(
                user_defined_fields.UD_ACCOUNT_NUMBER, user_defined_fields.FIELD_VALUE
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "ACCOUNT_NO")
            .withColumnRenamed("FIELD_VALUE", "AR_DOSEON_2ND_PERIOD")
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ============================= Join Conditions ======================================

        ProfitsAccount_UserDefinedFieldsUD1_Join_Condition = (
            user_defined_fields_UD1["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD2_Join_Condition = (
            user_defined_fields_UD2["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD3_Join_Condition = (
            user_defined_fields_UD3["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD4_Join_Condition = (
            user_defined_fields_UD4["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD5_Join_Condition = (
            user_defined_fields_UD5["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD6_Join_Condition = (
            user_defined_fields_UD6["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD7_Join_Condition = (
            user_defined_fields_UD7["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD8_Join_Condition = (
            user_defined_fields_UD8["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD9_Join_Condition = (
            user_defined_fields_UD9["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD10_Join_Condition = (
            user_defined_fields_UD10["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD11_Join_Condition = (
            user_defined_fields_UD11["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        ProfitsAccount_UserDefinedFieldsUD12_Join_Condition = (
            user_defined_fields_UD12["ACCOUNT_NO"]
            == loan_details_final["ACCOUNT_NUMBER"]
        )

        LoanDetails_Portfolio_Join_condition = (
            portfolio["unit_code_profits"] == loan_details_final["prft_acc_unit"]
        )

        # ==================== Apply the Joins =========================

        Joined_df = (
            loan_details_final.join(
                user_defined_fields_UD1,
                ProfitsAccount_UserDefinedFieldsUD1_Join_Condition,
                "inner",
            )
            .join(
                user_defined_fields_UD2,
                ProfitsAccount_UserDefinedFieldsUD2_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD3,
                ProfitsAccount_UserDefinedFieldsUD3_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD4,
                ProfitsAccount_UserDefinedFieldsUD4_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD5,
                ProfitsAccount_UserDefinedFieldsUD5_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD6,
                ProfitsAccount_UserDefinedFieldsUD6_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD7,
                ProfitsAccount_UserDefinedFieldsUD7_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD8,
                ProfitsAccount_UserDefinedFieldsUD8_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD9,
                ProfitsAccount_UserDefinedFieldsUD9_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD10,
                ProfitsAccount_UserDefinedFieldsUD10_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD11,
                ProfitsAccount_UserDefinedFieldsUD11_Join_Condition,
                "left",
            )
            .join(
                user_defined_fields_UD12,
                ProfitsAccount_UserDefinedFieldsUD12_Join_Condition,
                "left",
            )
            .join(portfolio, LoanDetails_Portfolio_Join_condition, "left")
        )

        # ================= Set the Keys for Hashkey================

        SplitBalanceId__primary_key_fields = [
            loan_details_final.ACCOUNT_NUMBER,
            lit("profits"),
        ]

        # =============== Creation of Hashkeys ===================

        target_table_profits = Joined_df.withColumn(
            "SplitBalanceId",
            sha2(
                concat_ws(";", *SplitBalanceId__primary_key_fields),
                256,
            ),
        )

        # =========== Selection of Final Columns ================

        result_df_profits = target_table_profits.select(
            target_table_profits.SplitBalanceId.cast("string").alias("SplitBalanceId"),
            lit("profits").cast("string").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            loan_details_final.ACCOUNT_NUMBER.cast("string").alias("SplitBalanceKey"),
            loan_details_final.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            user_defined_fields_UD2.SYN_YPOL_KEF.cast("string").alias(
                "TotalRemainingAmount"
            ),
            user_defined_fields_UD3.KEF_1ST_PERIOD.cast("string").alias(
                "FirstPeriodAmount"
            ),
            user_defined_fields_UD4.YPOL_KEF_1ST_PERIOD.cast("string").alias(
                "FirstPeriodRemainingAmount"
            ),
            user_defined_fields_UD5.START_DATE_1ST_PERIOD.cast("string").alias(
                "FirstPeriodStartDate"
            ),
            user_defined_fields_UD6.END_DATE_1ST_PERIOD.cast("string").alias(
                "FirstPeriodEndDate"
            ),
            user_defined_fields_UD7.AR_DOSEON_1ST_PERIOD.cast("string").alias(
                "FirstPeriodNumberOfInstallements"
            ),
            user_defined_fields_UD8.KEF_2ND_PERIOD.cast("string").alias(
                "SecondPeriodAmount"
            ),
            user_defined_fields_UD9.YPOL_KEF_2ND_PERIOD.cast("string").alias(
                "SecondPeriodRemainingAmount"
            ),
            user_defined_fields_UD10.START_DATE_2ND_PERIOD.cast("string").alias(
                "SecondPeriodStartDate"
            ),
            user_defined_fields_UD11.END_DATE_2ND_PERIOD.cast("string").alias(
                "SecondPeriodEndDate"
            ),
            user_defined_fields_UD12.AR_DOSEON_2ND_PERIOD.cast("string").alias(
                "SecondPeriodNumberOfInstallements"
            ),
            loan_details_final.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
             and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_split_balance_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
